# RE-ARC vs Original ARC Comparison

For a chosen task, shows the original ARC training pairs followed by
a sample of RE-ARC generated examples.

Set `TASK_ID` and `N_REARC` below, then Run All.

In [ ]:
%matplotlib inline
import json, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Config ────────────────────────────────────────────────────────────────────
TASK_ID  = '09629e4f'   # ← change to any task ID
N_REARC  = 8            # number of RE-ARC examples to show
SEED     = 42
# ─────────────────────────────────────────────────────────────────────────────

ROOT     = Path('..').resolve()
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

def show_pair(ax_in, ax_out, inp, out, label_in, label_out):
    for ax, grid, label in [(ax_in, inp, label_in), (ax_out, out, label_out)]:
        ax.imshow(np.array(grid), cmap=CMAP, norm=NORM, interpolation='nearest')
        ax.set_title(label, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

# Load original ARC task
arc_path  = ROOT / 'data' / 'training' / f'{TASK_ID}.json'
arc_task  = json.loads(arc_path.read_text())
orig_pairs = arc_task['train']

# Load RE-ARC examples
rearc_path = ROOT / 'data' / 're_arc' / f'{TASK_ID}.json'
rearc_all  = json.loads(rearc_path.read_text())
random.seed(SEED)
rearc_sample = random.sample(rearc_all, N_REARC)

print(f'Task: {TASK_ID}')
print(f'Original pairs : {len(orig_pairs)}')
print(f'RE-ARC total   : {len(rearc_all)}  (showing {N_REARC})')

In [ ]:
# ── Original ARC training pairs ───────────────────────────────────────────────
print('=' * 60)
print('  ORIGINAL ARC TRAINING PAIRS')
print('=' * 60)

fig, axes = plt.subplots(len(orig_pairs), 2,
                          figsize=(6, 3 * len(orig_pairs)), squeeze=False)
fig.patch.set_facecolor('#e8f0fe')
fig.suptitle(f'{TASK_ID} — Original ({len(orig_pairs)} pairs)', fontsize=10, y=1.01)

for i, pair in enumerate(orig_pairs):
    show_pair(axes[i,0], axes[i,1],
              pair['input'], pair['output'],
              f'Original {i+1} — input', f'Original {i+1} — output')

plt.tight_layout()
plt.show()

In [ ]:
# ── RE-ARC generated examples ─────────────────────────────────────────────────
print('=' * 60)
print(f'  RE-ARC GENERATED EXAMPLES  (random sample of {N_REARC})')
print('=' * 60)

fig, axes = plt.subplots(N_REARC, 2,
                          figsize=(6, 3 * N_REARC), squeeze=False)
fig.patch.set_facecolor('#e8fee8')
fig.suptitle(f'{TASK_ID} — RE-ARC ({N_REARC} of {len(rearc_all)})', fontsize=10, y=1.01)

for i, pair in enumerate(rearc_sample):
    inp = np.array(pair['input'])
    out = np.array(pair['output'])
    show_pair(axes[i,0], axes[i,1],
              inp, out,
              f'RE-ARC {i+1} — input  {inp.shape}',
              f'RE-ARC {i+1} — output  {out.shape}')

plt.tight_layout()
plt.show()

In [ ]:
# ── Grid size distribution in RE-ARC ─────────────────────────────────────────
from collections import Counter

in_shapes  = Counter(tuple(np.array(p['input']).shape)  for p in rearc_all)
out_shapes = Counter(tuple(np.array(p['output']).shape) for p in rearc_all)

print(f'RE-ARC input shapes  (top 10): {in_shapes.most_common(10)}')
print(f'RE-ARC output shapes (top 10): {out_shapes.most_common(10)}')
print()

orig_in_shapes  = [tuple(np.array(p['input']).shape)  for p in orig_pairs]
orig_out_shapes = [tuple(np.array(p['output']).shape) for p in orig_pairs]
print(f'Original input shapes : {orig_in_shapes}')
print(f'Original output shapes: {orig_out_shapes}')

# Colour usage
import matplotlib.pyplot as plt
rearc_colors  = Counter(int(v) for p in rearc_all
                         for row in p['input'] for v in row if v != 0)
orig_colors   = Counter(int(v) for p in orig_pairs
                         for row in p['input'] for v in row if v != 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, counts, title in [
    (axes[0], orig_colors,  'Original — colour freq (inputs, excl. black)'),
    (axes[1], rearc_colors, 'RE-ARC   — colour freq (inputs, excl. black)'),
]:
    cols = sorted(counts.keys())
    ax.bar(cols, [counts[c] for c in cols],
           color=[ARC_COLORS[c] for c in cols], edgecolor='grey')
    ax.set_xticks(cols)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Colour index')
plt.tight_layout()
plt.show()